In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Get_non_bonded_potential_energy)=
# Get non-bonded potential energy

*Calculating non-bonded (electrostatic and van der Waals) interaction energies between selections.*

The function {func}`molsysmt.molecular_mechanics.get_non_bonded_potential_energy` computes non-bonded interaction energies across molecular interfaces or evaluates group-by-group interaction energy matrices.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.molecular_mechanics.get_non_bonded_potential_energy`.
:::


:::{note}
:class: dropdown

**Molecular Mechanics Backend Engine**

Molecular mechanics and force evaluations in MolSysMT are executed using backend simulation engines (currently **OpenMM** by default). MolSysMT intentionally unifies the Pythonic interface and data structures while delegating energy evaluations, parameterizations, and gradient calculations to high-performance, battle-tested molecular simulation engines.
:::

Execution platform names belong to the selected engine. OpenMM accepts `Reference`, `CPU`, `CUDA`, `OpenCL` and installed plugin names, subject to registration and device readiness. The `platform` argument controls newly constructed contexts; an existing Context/Simulation retains its platform. A requested platform is never replaced after failure. See {ref}`OpenMM platform discovery <Tutorial_OpenMM_Available_platforms>` for registration and distinct error outcomes.


## Basic usage

Let's compute the non-bonded interaction energy across a protein–protein complex interface:


In [2]:
import molsysmt as msm
import pyunitwizard as puw
import matplotlib.pyplot as plt
import numpy as np


In [3]:
molsys = msm.convert(msm.systems['Barnase-Barstar']['barnase_barstar.h5msm'])


:::{note}
:class: dropdown

**Demo Systems Catalog**

This tutorial uses demonstration datasets provided by MolSysMT. To explore the full catalog of bundled systems, forms, and file paths, visit the {ref}`Demo Systems <user-foundations-entrance-demo-systems>` guide.
:::


We calculate the total non-bonded interaction energy between Barnase and Barstar:


In [4]:
u_nb = msm.molecular_mechanics.get_non_bonded_potential_energy(molsys,
                                                              selection='molecule_name=="BARNASE"',
                                                              selection_2='molecule_name=="BARSTAR"')
print('Total non-bonded interaction energy:', u_nb)


Total non-bonded interaction energy: -3162.2924976349714 kilojoule / mole


## Pairwise interaction energy matrix between atom sets

Lists of atom selections define ordered matrix rows and columns. We use the first three groups of each protein to keep this tutorial bounded. Each entry is the non-bonded interaction energy between one Barnase group and one Barstar group. Select the full lists without `[:3]` to request the full protein matrix; its dense output and calculation grow with both partition counts.


In [5]:
barnase_groups = msm.select(molsys, selection='all in groups of molecule_name=="BARNASE"')[:3]
barstar_groups = msm.select(molsys, selection='all in groups of molecule_name=="BARSTAR"')[:3]

u_matrix = msm.molecular_mechanics.get_non_bonded_potential_energy(molsys, selection=barnase_groups, selection_2=barstar_groups)
u_vals = puw.get_value(u_matrix, to_unit='kJ/mol')
print('Selected group matrix shape:', u_vals.shape)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(u_vals, origin='lower', cmap='coolwarm', vmin=-125, vmax=125, aspect='auto')
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Interaction Energy (kJ/mol)')
ax.set_title('Selected Group Non-bonded Interaction Energy Matrix')
ax.set_xlabel('Barstar Selected Group Position')
ax.set_ylabel('Barnase Selected Group Position')
plt.tight_layout()
plt.show()


Selected group matrix shape: (3, 3)


We can sum each row or column to obtain contributions from the **selected partner groups**. These subset sums are not whole-protein contributions. If you request repeated or overlapping partitions within one axis, summing entries can count atoms more than once.


In [6]:
u_barnase_res = puw.get_value(u_matrix.sum(axis=1), to_unit='kJ/mol')
u_barstar_res = puw.get_value(u_matrix.sum(axis=0), to_unit='kJ/mol')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharey=True)
ax1.bar(range(len(u_barnase_res)), u_barnase_res, color='steelblue')
ax1.set_title('Selected Barnase Group Energies with Selected Barstar Groups')
ax1.set_xlabel('Barnase Selected Group Position')
ax1.set_ylabel('Energy (kJ/mol)')
ax1.grid(True, linestyle='--', alpha=0.4)

ax2.bar(range(len(u_barstar_res)), u_barstar_res, color='coral')
ax2.set_title('Selected Barstar Group Energies with Selected Barnase Groups')
ax2.set_xlabel('Barstar Selected Group Position')
ax2.set_ylabel('Energy (kJ/mol)')
ax2.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


## Mixing flat and partitioned selections

Either axis can contain one atom set or an explicit list of sets. A singleton partition remains a matrix axis.

| `selection` | `selection_2` | Result shape |
| :--- | :--- | :--- |
| Flat set | Flat set | Scalar |
| List of p sets | Flat set | `(p, 1)` |
| Flat set | List of q sets | `(1, q)` |
| List of p sets | List of q sets | `(p, q)` |

The mixed result below must match the first column of the matrix above. Rows retain `selection` order; columns retain `selection_2` order.


In [7]:
u_mixed = msm.molecular_mechanics.get_non_bonded_potential_energy(
    molsys, selection=barnase_groups, selection_2=barstar_groups[0])
print('Mixed selection shape:', u_mixed.shape)
np.testing.assert_allclose(puw.get_value(u_mixed, to_unit='kJ/mol'), u_vals[:, :1], rtol=1e-6, atol=1e-5)


Mixed selection shape: (3, 1)


An empty atom set gives a zero energy with units: `[]` is one flat empty set, while `[[], []]` requests two empty partitions. Repeated atom indices within a set are deduplicated; repeated partitions and shared atoms within one axis are retained. Atoms shared **between** the two axes raise `ArgumentConflictError`, because the current disjoint-set decomposition does not define that case. Without `selection_2`, only one flat selection is accepted and its internal non-bonded energy is calculated.

The result follows the active PyUnitWizard energy policy. Use an explicit `to_unit` when exporting numbers. Evaluation uses a copied engine System and preserves source selections, coordinates, parameters and force groups.

This calculation includes the standard OpenMM `NonbondedForce` and its exceptions, excluding implicit-solvent and custom-force contributions. A model without exactly one standard non-bonded force is rejected explicitly. The result is a model-dependent interaction energy, not a binding free energy or a prediction of binding stability. Periodic energies retain the configured engine method and current box; the partition controls use bounded nonperiodic models and do not qualify every periodic/custom model. Extract one structure explicitly for analysis: the function does not loop over a structure axis, and supplied Context/Simulation inputs use current positions.


## Visualizing group interaction energies in 3D

A complete group analysis can be projected onto the protein in 3D. The retained pre-generated illustration below shows that visualization workflow; it does not display the three-by-three subset calculated above.


In [8]:
# all_res_vals = list(u_barnase_res) + list(u_barstar_res)
# view = msm.view(molsys)
# view.whole.set_color_by_values(all_res_vals, element='group', palette='coolwarm')
# view


```{raw} html
<iframe src="../../../../_static/views/tools_mm_get_non_bonded_potential_energy_1.html" width="100%" height="450px" frameborder="0"></iframe>
```


:::{seealso}
:class: dropdown

**Related Tools & References**

- {ref}`Convert <Tutorial_Convert>`: Convert molecular systems with {func}`molsysmt.basic.convert`.
- {ref}`Select <Tutorial_Select>`: Query atom selections with {func}`molsysmt.basic.select`.
- {ref}`Get potential energy <Tutorial_Get_potential_energy>`: Calculate total potential energy with {func}`molsysmt.molecular_mechanics.get_potential_energy`.
:::
